In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wafaaelhusseini/e-commerce-transactions-clickstream")

print("Path to dataset files:", path)

c:\Users\Usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 11.1M/11.1M [00:03<00:00, 3.11MB/s]

Extracting files...


Path to dataset files: C:\Users\Usuario\.cache\kagglehub\datasets\wafaaelhusseini\e-commerce-transactions-clickstream\versions\1


In [ ]:
import os
import pandas as pd

# Listar archivos del dataset
print(os.listdir(path))

# Cargar cada CSV en un DataFrame
customers = pd.read_csv(os.path.join(path, "customers.csv"))
events = pd.read_csv(os.path.join(path, "events.csv"))
order_items = pd.read_csv(os.path.join(path, "order_items.csv"))
orders = pd.read_csv(os.path.join(path, "orders.csv"))
products = pd.read_csv(os.path.join(path, "products.csv"))
reviews = pd.read_csv(os.path.join(path, "reviews.csv"))
sessions = pd.read_csv(os.path.join(path, "sessions.csv"))

events.head()


# Análisis de Productos y Ventas

Análisis de `products` (catálogo) y `order_items` (líneas de venta): categorías, precios, márgenes y top ventas.

In [ ]:
import matplotlib.pyplot as plt

pd.set_option('display.width', 140)

# Margen en % por producto
products['margin_pct'] = products['margin_usd'] / products['price_usd'] * 100

print("Productos por categoría:")
print(products['category'].value_counts())


## Precios por categoría

In [ ]:
price_by_cat = products.groupby('category')['price_usd'].agg(['count', 'mean', 'min', 'max']).sort_values('mean', ascending=False)
print(price_by_cat)

price_by_cat['mean'].plot(kind='bar', figsize=(8, 4), title='Precio promedio por categoría (USD)')
plt.ylabel('USD')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## Márgenes por categoría

In [ ]:
margin_by_cat = products.groupby('category').agg(
    margin_usd_mean=('margin_usd', 'mean'),
    margin_pct_mean=('margin_pct', 'mean')
).sort_values('margin_pct_mean', ascending=False)
print(margin_by_cat)

print()
print("Margen global:")
print(f"  Precio promedio: ${products['price_usd'].mean():.2f}")
print(f"  Costo promedio:  ${products['cost_usd'].mean():.2f}")
print(f"  Margen promedio: ${products['margin_usd'].mean():.2f} ({products['margin_pct'].mean():.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
margin_by_cat['margin_usd_mean'].plot(kind='bar', ax=axes[0], title='Margen promedio ($) por categoría')
margin_by_cat['margin_pct_mean'].plot(kind='bar', ax=axes[1], title='Margen promedio (%) por categoría', color='orange')
for ax in axes:
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


## Top ventas (uniendo `order_items` con `products`)

In [ ]:
sales = order_items.groupby('product_id').agg(
    qty=('quantity', 'sum'),
    revenue=('line_total_usd', 'sum')
).reset_index()
sales = sales.merge(products, on='product_id')
sales['total_margin'] = sales['margin_usd'] * sales['qty']

top_qty = sales.sort_values('qty', ascending=False).head(10)
print("Top 10 por unidades vendidas:")
print(top_qty[['product_id', 'name', 'category', 'qty', 'revenue']].to_string(index=False))

top_qty.set_index('name')['qty'].plot(kind='barh', figsize=(8, 5), title='Top 10 productos por unidades vendidas')
plt.xlabel('Unidades')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
top_rev = sales.sort_values('revenue', ascending=False).head(10)
print("Top 10 por ingresos:")
print(top_rev[['product_id', 'name', 'category', 'qty', 'revenue']].to_string(index=False))

top_rev.set_index('name')['revenue'].plot(kind='barh', figsize=(8, 5), title='Top 10 productos por ingresos (USD)', color='green')
plt.xlabel('Ingresos (USD)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## Ventas e ingresos totales por categoría

In [ ]:
cat_summary = sales.groupby('category').agg(
    unidades=('qty', 'sum'),
    ingresos=('revenue', 'sum'),
    margen_total=('total_margin', 'sum')
).sort_values('ingresos', ascending=False)
print(cat_summary)

cat_summary[['ingresos', 'margen_total']].plot(kind='bar', figsize=(9, 5), title='Ingresos y margen total por categoría')
plt.ylabel('USD')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
